# QEFM Plotting Notebook

## How to set up the Python environment

To run this notebook, you will need to do the following:

1) Run the setup script to create the python environment.
```bash
bash setup.sh
```
*Note: if you run into issues, try deleting your environment and cache using 
```rm -rf path/to/.venv /home/sagemaker-user/.cache```, then re-running the setup.sh file.*

2) Select the python environment called "Python (Weather FM) from the top right of the JupyterHub interface.
3) Run the notebook through, changing the associated variables in the "config" section below. Do not change anything else. 

## Purpose of this notebook
This notebook is used to generate visualizations of various statistics for Weather Foundation Models. This is done by loading postprocessed forecast model rollouts, which have various statistics computed. 

The notebook begins with some basic python imports, and then contains a user configuration setting. See that markdown section for in-depth info on what parts to edit, and what different variables mean. The notebook then loads filenames from a remote filesystem, and starts loading the model statistics by type: regional and global. Regional statistics are limited to a certain part of Earth; for example, you can load statistics for the Northern Hemisphere, or the Tropics. See the user configuration section markdown for more info on these regions. Global statistics are similar statistics to the regional ones, but for the whole planet. 

Once the global and regional stats have been loaded into memory, we can visualize how our models perform over time, for different variables and pressure levels. This is done using the respective plotting functions, to plot regional and global stats. Plots will be saved to the output directory in the same directory as this notebook. 

## More on the statistics
The statistics we calculate are fairly simple. Statistics are calculated by latitude and longitude, by lead time, by pressure level, and by variable. Here's some more info on what those terms mean:

### Latitude and Longitude
Global coordinates that determine where the statistic takes place. A latitude and longitude of 34.7295° N, 86.5853° W would be Huntsville, AL, for example. For statistics purposes, we have coarsened the resolution of the latitude and longitude grid to have 91 latitudes and 144 longitudes, meaning all values will be integer values.

### Lead Time
This is the number of hours after the initial conditions; so a lead time of 24 hours is a one-day weather prediction.

### Pressure Level
Measured in hectopascals (hPa) or millibars (mb), this dictates how close or far from sea level our weather data is taken from. 1000 hPa is at sea level, while 50 hPa is very far up in the atmosphere.

### Variable
Our statistics are calculated across 2 sets of variables: 3D variables, which are measured at 13 different pressure levels, and 2D/slice variables, which are measured at a single pressure level.

#### 3D Variables
- **T**: temperature (in degrees Kelvin).
- **Q**: specific humidity; this is the mass of water vapor in a parcel of air divided by the total mass of that air. This is used to track moisture content in the atmosphere for predicting precipitation, clouds, and storm intensity. This is measured in grams per kilogram. 
- **U**: u-component of wind, or the "east-west" component (in meters/second).
- **V**: v-component of wind, or the "north-south" component (in meters/second).
- **Z**: geopotential, this is the energy required to lift 1 kilogram of air from sea level to a specific height in the atmosphere. In other words, it's the "gravitational energy" per kilogram of air. It's used to map pressure surfaces and identify weather patterns like high/low pressure systems. This is measured in meters squared per second squared (m²/s²). 

#### 2D Variables/Slices
For our statistics, we are using 3D variable slices as our 2D variables; these are 3D variables taken at a specific height (such as 10 meters). Our slice variables include: 
- **T2m**: Temperature at 2m (in degrees Kelvin).
- **U10m/V10m**: U and V components of wind, at 10 meters.
- **D2m**: Dew Point at 2 meters; this is the temperature to which air must cool for water vapor to condense into dew, and is used to measure atmospheric moisture and assess how humid or muggy it feels outside.

### A note on the term "model"
Below you'll find explanations of how the global and regional statistics are calculated for a weather foundation model. To calculate these stats, meteorologists use 2 other types of models:
- **Reanalysis models**: ...
- **Climatology models**: ...

### Global stats
These stats are displayed as global heatmaps, in 2 rows. The first row will be the forecast model compared with its re-analysis model, while the second row will be the forecast model compared to the GEOS-FP model. (need Kathy's input on this) 
- **Forecast difference/"fcst diff"**: this is the global difference in predictions of a certain variable, taken at a certain lead time and pressure level. In the plots, you will see this as [Experimental Model] - [Control Model]; this is essentially seeing how good the model is at recreating the actual measured variable value at that specific time/location/etc. A lower absolute value is best here; a highly negative value means the model under-predicts that variable, while a highly positive value means the model over-predicts that variable. 
- **Anomaly Correlation/ACC**: ...
- **Root Mean Squared Error/RMSE**: ...

### Regional stats
These stats are calculated by region, over the entire forecast lead time. For our datasets, we used a period of 7 days for each model, so the regional stats will be a line plot of how both ACC and RMSE change over time. For visual clarity, we chose to display 2 models: the weather foundation model of choice, and the GEOS-FP model (need more from Kathy). 
- **Anomaly Correlation/ACC**: see global stats explanation.  
- **Root Mean Squared Error/RMSE**: see global stats explanation.  

## Imports

In [1]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import xarray as xr
from glob import glob
from pathlib import Path
import yaml
from tqdm import tqdm
import sys
import os
from datetime import datetime

sys.path.insert(0, "Plots")

# Import refactored plotting functions
from plots.load_stats import load_regional_stats_data, load_global_stats_data
from plots.plots_regional import create_regional_plots
from plots.plots_global import create_global_plots
from plots.utils_discover import get_stat_dataset_filenames, create_base_patterns

## Config
Levels: [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50] hPa

Variables:
- 3D variables: H, Q, T, U, V
- 2D variables: P, PS
- Slice variables: T2m, U10m, V10m, D2m

Regions:
- NHE: Northern Hemisphere (20°N to 80°N)
- SHE: Southern Hemisphere (80°S to 20°S)
- TRO: Tropical Belt (20°S to 20°N)

Start month: either May or Dec 2024 (just input "May" or "Dec"). 
**If you pick a value outside of this range, the script will fail, as our data is only computed for this date range.**

In [2]:
# Stats are calculated for the full month of May/Dec 2024
# Month is not case-sensitive
MONTH = "Dec"

# We will look for stats in this directory; stats are stored in netcdf files
FORECAST_MODEL = "AIFS"  # Choose from 'AIFS' or 'Prithvi'

# Will only plot these variables across 3D and 2D collections
VARS_TO_PLOT = ["Z", "Q", "T", "U", "V", "D2m", "T2m", "V10m", "U10m"]

# Will only create level plots using these levels (zonal plots will still be created with all levels)
LEVELS_TO_PLOT = [925, 850, 500, 200]

# Will only plot leads from this list; forecasts were made for up to 7 days
# Leads are in hours
LEADS_TO_PLOT = [24, 72, 120, 168]

## Get filenames of statistics netCDFs
These have already been created.

In [3]:
# Create list of all statistics filenames across all experiments
exp_patterns = create_base_patterns(FORECAST_MODEL)
all_stat_filenames = get_stat_dataset_filenames(exp_patterns, MONTH)
regional_stat_filenames = all_stat_filenames["regional"]
global_stat_filenames = all_stat_filenames["global"]

# There should be 1 file per experiment in each of regional, global filename lists
regional_file_str = "\n".join(map(str, regional_stat_filenames))
global_file_str = "\n".join(map(str, global_stat_filenames))
print(regional_file_str)
print(global_file_str)

['/discover/nobackup/ajkerr1/qefm_qa/s3_bucket/dec_2024/stats_regional_f5295fp_GEOSFP_MERRA2_20241201-20241231_len7d_int12h_spc1d_91x144.nc4']
['/discover/nobackup/ajkerr1/qefm_qa/s3_bucket/dec_2024/stats_regional_ERA5_ERA5_ERA5_20241201-20241231_len7d_int12h_spc1d_91x144.nc4']
['/discover/nobackup/ajkerr1/qefm_qa/s3_bucket/dec_2024/stats_regional_AIFS_GEOSFP_MERRA2_20241201-20241231_len7d_int12h_spc1d_91x144.nc4']
['/discover/nobackup/ajkerr1/qefm_qa/s3_bucket/dec_2024/stats_regional_AIFS_ERA5_ERA5_20241201-20241231_len7d_int12h_spc1d_91x144.nc4']
['/discover/nobackup/ajkerr1/qefm_qa/s3_bucket/dec_2024/stats_global_f5295fp_GEOSFP_MERRA2_20241201-20241231_len7d_int12h_spc1d_91x144.nc4']
['/discover/nobackup/ajkerr1/qefm_qa/s3_bucket/dec_2024/stats_global_ERA5_ERA5_ERA5_20241201-20241231_len7d_int12h_spc1d_91x144.nc4']
['/discover/nobackup/ajkerr1/qefm_qa/s3_bucket/dec_2024/stats_global_AIFS_GEOSFP_MERRA2_20241201-20241231_len7d_int12h_spc1d_91x144.nc4']
['/discover/nobackup/ajkerr1/qef

## Load stats data from the cloud

### Regional

In [4]:
regional_data = load_regional_stats_data(regional_stat_filenames, vars_to_fetch=VARS_TO_PLOT)

Starting loading of regional stat data from netcdf...


Loading data...: 100%|██████████| 4/4 [00:00<00:00, 15.49it/s]


✓ Loading complete: 4 experiments, 31 dates


### Global

In [5]:
global_data_comp = load_global_stats_data(global_stat_filenames, vars_to_fetch=VARS_TO_PLOT)

Loading data...: 100%|██████████| 4/4 [02:05<00:00, 31.47s/it]


✓ Loading complete: 4 experiments, 31 dates


# Stat Plotting

## Make output directory
This is where the plots will go, separated into global and regional plots. 

In [6]:
output_dir = Path(f"plots/outputs/{FORECAST_MODEL}")
output_dir.mkdir(exist_ok=True, parents=True)

## Regional stat plotting

In [7]:
# Creates only per-level plots of ACC/RMSE
create_regional_plots(
    regional_data,
    vars_to_plot=VARS_TO_PLOT,
    levels_to_plot=LEVELS_TO_PLOT,
    plots_dir=output_dir
)


Making level plots for Q (3D) NHE:

Making level plots for T (3D) NHE:

Making level plots for U (3D) NHE:

Making level plots for V (3D) NHE:

Making level plots for Z (3D) NHE:

Making plot for D2m (2D) NHE

Making plot for T2m (2D) NHE

Making plot for U10m (2D) NHE

Making plot for V10m (2D) NHE

Making level plots for Q (3D) TRO:

Making level plots for T (3D) TRO:

Making level plots for U (3D) TRO:

Making level plots for V (3D) TRO:

Making level plots for Z (3D) TRO:

Making plot for D2m (2D) TRO

Making plot for T2m (2D) TRO

Making plot for U10m (2D) TRO

Making plot for V10m (2D) TRO

Making level plots for Q (3D) SHE:

Making level plots for T (3D) SHE:

Making level plots for U (3D) SHE:

Making level plots for V (3D) SHE:

Making level plots for Z (3D) SHE:

Making plot for D2m (2D) SHE

Making plot for T2m (2D) SHE

Making plot for U10m (2D) SHE

Making plot for V10m (2D) SHE

PLOTTING COMPLETE!
Plots saved to: plots/outputs/AIFS/regional

All finished! :)



## Global stat plotting

In [8]:
create_global_plots(
    global_data_comp,
    vars_to_plot=VARS_TO_PLOT, 
    levels_to_plot=LEVELS_TO_PLOT,
    leads_to_plot=LEADS_TO_PLOT,
    output_dir=output_dir
)

Extracting data from dictionary...

GENERATING PLOTS

Making plots for Q (index 0, 3D)...
Making plots for T (index 1, 3D)...
Making plots for U (index 2, 3D)...
Making plots for V (index 3, 3D)...
Making plots for Z (index 4, 3D)...
Making plots for D2m (index 0, 2D)...
Making plots for T2m (index 1, 2D)...
Making plots for U10m (index 2, 2D)...
Making plots for V10m (index 3, 2D)...

PLOTTING COMPLETE!
Plots saved to: plots/outputs/AIFS/global

All finished! :)

